
# Dear Canada — Spatial & Character Extraction Pipeline

This notebook implements the pipeline agreed in planning:

1. **NER recall net** — spaCy pass over each book to generate candidate person
   and location surface forms.
2. **Candidate list review** — LLM-assisted (or manual) cleanup of the NER
   candidate lists into canonical entities.
3. **Registries** — a per-book `CharacterRegistry` and `LocationRegistry` that
   track canonical entities, aliases, and (for characters) relation to the
   narrator. These are maintained by *code*, not by the LLM, and are updated
   incrementally as entries are processed in order.
4. **Sliding-window LLM extraction** — one entry at a time, in narrative
   order, with a compact `state_summary` (built from the registries + last
   known narrator location) injected into the prompt. The model is asked to
   report only what *this entry* states or clearly implies, and to leave
   fields `null` otherwise — continuity/"still there" reasoning is handled
   downstream in code, not by the model.
5. **Grounding validation** — every `evidence` quote returned by the model is
   checked against the source entry text (exact + fuzzy match). Entries with
   ungrounded evidence are flagged for review rather than silently trusted.
6. **Carry-forward post-processing** — deterministic fill-forward of `null`
   locations using the last explicitly-stated value.
7. **Export** — a full debug dataset (with evidence + grounding flags, for
   your manual-label evaluation) and a lean public dataset (evidence
   stripped, for the site/downstream geocoding).

**Before running on the full corpus:** run this on your 2 pilot books first,
inspect the QA report (`qa_report.json`) and the registries, and compare a
sample of entries against your manual gold labels. See the notes at the end
of the notebook for what to check specifically.


## 0. Setup & configuration

In [ ]:

import os
import re
import json
import glob
import time
import difflib
from dataclasses import dataclass, field, asdict
from pathlib import Path

import pandas as pd

# --- paths -------------------------------------------------------------
DATA_DIR = Path("./data")            # expects one CSV per book: book,chapter_id,entry_id,date,entry_text
OUT_DIR = Path("./output")
DEBUG_DIR = OUT_DIR / "debug"
REGISTRY_DIR = OUT_DIR / "registries"
CANDIDATE_DIR = OUT_DIR / "candidate_lists"   # NER candidate lists for manual/LLM review

for d in (OUT_DIR, DEBUG_DIR, REGISTRY_DIR, CANDIDATE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- pilot scope ---------------------------------------------------------
# Fill in with the CSV filenames (without extension) of your 2 pilot books.
PILOT_BOOKS = [
    "SofS",   # <- replace with your actual book codes
    "example_book_2",
]

# --- LLM config ------------------------------------------------------------
# Set your key as an environment variable before running:
#   export ANTHROPIC_API_KEY=sk-ant-...
import anthropic
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env

# Model choice: check https://docs.claude.com/en/docs/about-claude/models/overview
# for the current lineup before running at full-corpus scale, since model
# names/pricing change over time. A mid-tier model is normally sufficient
# for this kind of structured extraction; reserve a larger model for entries
# that fail grounding checks on the first pass.
MODEL_NAME = "claude-sonnet-5"

# How many prior entries (full text) to include as context for the sliding
# window, in addition to the compact state_summary. This is for resolving
# pronouns/references, not for the model to infer continuity from — the
# state_summary + carry-forward logic downstream handles continuity.
CONTEXT_WINDOW_ENTRIES = 2



## 1. Load book CSVs

Expects columns: `book, chapter_id, entry_id, date, entry_text`, one CSV per
book. Entries are sorted into narrative order (chapter, then entry) since the
whole pipeline depends on processing entries sequentially.


In [ ]:

def load_book(book_code: str, data_dir: Path = DATA_DIR) -> pd.DataFrame:
    path = data_dir / f"{book_code}.csv"
    df = pd.read_csv(path)
    required_cols = {"book", "chapter_id", "entry_id", "date", "entry_text"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {missing}")
    # Sort into narrative order. Adjust the sort key if chapter/entry ids
    # aren't naturally sortable as strings (e.g. "ch1", "ch2", ..., "ch10").
    df["_chapter_num"] = df["chapter_id"].str.extract(r"(\d+)").astype(int)
    df["_entry_num"] = df["entry_id"].str.extract(r"e(\d+)$").astype(int)
    df = df.sort_values(["_chapter_num", "_entry_num"]).reset_index(drop=True)
    return df

def load_pilot_books(book_codes=PILOT_BOOKS) -> dict:
    return {b: load_book(b) for b in book_codes}

# books = load_pilot_books()
# books["SofS"].head()



## 2. NER recall net

A first, cheap pass over the whole book with spaCy to surface candidate
person and place names. This is a *recall* tool: its job is to make sure
nothing gets missed, not to make the final call on entity type or identity —
that happens in review (step 3) and in the LLM extraction itself.

Install the model once: `python -m spacy download en_core_web_trf`
(`en_core_web_sm` also works and is much faster/smaller if `en_core_web_trf`
isn't available; expect somewhat lower recall on non-English/period names).


In [ ]:

import spacy

try:
    nlp = spacy.load("en_core_web_trf")
except OSError:
    nlp = spacy.load("en_core_web_sm")

PERSON_LABELS = {"PERSON"}
PLACE_LABELS = {"GPE", "LOC", "FAC"}

def ner_pass_book(df: pd.DataFrame) -> dict:
    """Run NER over every entry in a book; return candidate name/location
    lists with counts and an example entry_id for each surface form."""
    names, places = {}, {}
    for _, row in df.iterrows():
        doc = nlp(row["entry_text"])
        for ent in doc.ents:
            target = names if ent.label_ in PERSON_LABELS else (
                places if ent.label_ in PLACE_LABELS else None)
            if target is None:
                continue
            key = ent.text.strip()
            if not key:
                continue
            if key not in target:
                target[key] = {"count": 0, "example_entry_id": row["entry_id"], "label": ent.label_}
            target[key]["count"] += 1
    return {"candidate_names": names, "candidate_places": places}

def save_candidate_lists(book_code: str, candidates: dict):
    path = CANDIDATE_DIR / f"{book_code}_candidates.json"
    with open(path, "w") as f:
        json.dump(candidates, f, indent=2)
    print(f"Wrote {path} — {len(candidates['candidate_names'])} name forms, "
          f"{len(candidates['candidate_places'])} place forms.")
    return path

# Example:
# candidates = ner_pass_book(books["SofS"])
# save_candidate_lists("SofS", candidates)



## 3. Review candidate lists

Two options, either is fine at pilot scale — pick per book depending on how
messy the NER output looks:

- **Manual**: open the JSON in `output/candidate_lists/`, and hand-edit it
  into a reviewed file (`*_reviewed.json`) that groups surface forms into
  canonical entities, e.g. `{"marie": ["Marie", "my sister"], ...}`.
- **LLM-assisted**: use `suggest_canonical_entities()` below to get a first
  pass at clustering/cleaning, then hand-review the suggestion before
  treating it as ground truth. Do not skip the human review step — this is
  exactly the kind of coreference decision that's cheap to get right now and
  expensive to unwind later in the registries.


In [ ]:

CANONICALIZE_SYSTEM_PROMPT = """You are cleaning up a raw named-entity list \
extracted from one chapter-book of a children's historical fiction diary \
novel. You will be given two flat lists: candidate PERSON surface forms and \
candidate PLACE surface forms, each with an occurrence count.

Your job: group surface forms that refer to the same real-world entity, and \
flag anything that looks like a false positive (e.g. a place name mistagged \
as a person, a common noun mistagged as a place, a narrator referring to \
herself in the diary's voice).

Return ONLY valid JSON, no preamble, in this shape:
{
  "characters": [
    {"suggested_id": "marie", "canonical_label": "Marie", "aliases": ["Marie", "my sister"]}
  ],
  "locations": [
    {"suggested_id": "grosse_ile", "canonical_label": "Grosse Ile quarantine station", "aliases": ["Grosse Isle", "the island"]}
  ],
  "likely_false_positives": ["..."]
}
"""

def suggest_canonical_entities(candidates: dict) -> dict:
    user_content = json.dumps({
        "candidate_names": {k: v["count"] for k, v in candidates["candidate_names"].items()},
        "candidate_places": {k: v["count"] for k, v in candidates["candidate_places"].items()},
    }, indent=2)

    resp = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        system=CANONICALIZE_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_content}],
    )
    text = resp.content[0].text.strip()
    text = re.sub(r"^```(json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    return json.loads(text)

def save_reviewed_suggestion(book_code: str, suggestion: dict):
    path = CANDIDATE_DIR / f"{book_code}_reviewed_DRAFT.json"
    with open(path, "w") as f:
        json.dump(suggestion, f, indent=2)
    print(f"Wrote DRAFT to {path} — review and rename to *_reviewed.json before use.")
    return path

def load_reviewed_entities(book_code: str) -> dict:
    path = CANDIDATE_DIR / f"{book_code}_reviewed.json"
    with open(path) as f:
        return json.load(f)

# Example:
# suggestion = suggest_canonical_entities(candidates)
# save_reviewed_suggestion("SofS", suggestion)
# (hand-review the _DRAFT file, save as *_reviewed.json, then:)
# reviewed = load_reviewed_entities("SofS")



## 4. Registries

`CharacterRegistry` and `LocationRegistry` hold canonical entities and are
updated as entries are processed, in order. They are seeded from the
reviewed candidate lists (step 3), and grow if the LLM extraction surfaces
a genuinely new entity mid-book (flagged, not silently added — see
`register_or_flag`).

`relation_to_narrator` lives here, once per character, rather than being
repeated (and risking drift) in every per-entry `character_mentions` object.


In [ ]:

@dataclass
class CharacterEntry:
    character_id: str
    canonical_label: str
    aliases: list = field(default_factory=list)
    relation_to_narrator: str = "unknown"   # e.g. sister, mother, friend, historical_figure, self
    first_mentioned_entry: str = None
    last_known_location: str = None
    last_location_stated_entry: str = None

@dataclass
class LocationEntry:
    location_id: str
    canonical_label: str
    aliases: list = field(default_factory=list)
    first_mentioned_entry: str = None
    coordinates: list = None   # filled in a later, separate geocoding pass

class CharacterRegistry:
    def __init__(self):
        self._by_id = {}
        self._alias_to_id = {}
        self.pending_review = []   # entities the LLM proposed that weren't in the reviewed list

    def seed_from_reviewed(self, reviewed_characters: list):
        for c in reviewed_characters:
            entry = CharacterEntry(
                character_id=c["suggested_id"],
                canonical_label=c["canonical_label"],
                aliases=list(c.get("aliases", [])),
                relation_to_narrator=c.get("relation_to_narrator", "unknown"),
            )
            self._by_id[entry.character_id] = entry
            for a in entry.aliases + [entry.canonical_label]:
                self._alias_to_id[a.lower()] = entry.character_id

    def resolve(self, label: str):
        return self._by_id.get(self._alias_to_id.get(label.lower()))

    def register_or_flag(self, label: str, entry_id: str, relation_hint: str = "unknown"):
        """Called when the LLM mentions a character not found by resolve().
        Auto-creates a provisional entry AND logs it for manual review —
        never silently trust a mid-book LLM-invented identity as final."""
        existing = self.resolve(label)
        if existing:
            return existing
        new_id = re.sub(r"[^a-z0-9]+", "_", label.lower()).strip("_")
        entry = CharacterEntry(
            character_id=new_id, canonical_label=label,
            aliases=[label], relation_to_narrator=relation_hint,
            first_mentioned_entry=entry_id,
        )
        self._by_id[new_id] = entry
        self._alias_to_id[label.lower()] = new_id
        self.pending_review.append({"character_id": new_id, "label": label, "entry_id": entry_id})
        return entry

    def update_location(self, character_id: str, location_label: str, entry_id: str):
        if character_id in self._by_id and location_label:
            self._by_id[character_id].last_known_location = location_label
            self._by_id[character_id].last_location_stated_entry = entry_id

    def export(self) -> list:
        return [asdict(v) for v in self._by_id.values()]


class LocationRegistry:
    def __init__(self):
        self._by_id = {}
        self._alias_to_id = {}
        self.pending_review = []

    def seed_from_reviewed(self, reviewed_locations: list):
        for l in reviewed_locations:
            entry = LocationEntry(
                location_id=l["suggested_id"],
                canonical_label=l["canonical_label"],
                aliases=list(l.get("aliases", [])),
            )
            self._by_id[entry.location_id] = entry
            for a in entry.aliases + [entry.canonical_label]:
                self._alias_to_id[a.lower()] = entry.location_id

    def resolve(self, label: str):
        if not label:
            return None
        return self._by_id.get(self._alias_to_id.get(label.lower()))

    def canonical_label(self, label: str) -> str:
        entry = self.resolve(label)
        return entry.canonical_label if entry else label

    def register_or_flag(self, label: str, entry_id: str):
        existing = self.resolve(label)
        if existing:
            return existing
        new_id = re.sub(r"[^a-z0-9]+", "_", label.lower()).strip("_")
        entry = LocationEntry(location_id=new_id, canonical_label=label,
                               aliases=[label], first_mentioned_entry=entry_id)
        self._by_id[new_id] = entry
        self._alias_to_id[label.lower()] = new_id
        self.pending_review.append({"location_id": new_id, "label": label, "entry_id": entry_id})
        return entry

    def export(self) -> list:
        return [asdict(v) for v in self._by_id.values()]



## 5. Basic context (per book, hand-authored)

Short, factual synopsis fed to the model to disambiguate references — not to
be used as a source of facts about individual entries. `narrative_structure`
matters: `"retrospective"` books (diary written after the fact, from a
different place/time than the events described) need extra scrutiny during
your pilot, since `narrator_location_w_writing` should reflect where the
narrator is *holding the pen*, which can differ sharply from the place/time
being narrated.


In [ ]:

BASIC_CONTEXT = {
    "SofS": {
        "narrator_name": "REPLACE_ME",
        "narrator_age_at_writing": None,
        "setting_summary": "REPLACE_ME — one or two factual sentences, no interpretation",
        "narrative_structure": "real_time",   # "real_time" | "retrospective"
        "instructions_note": (
            "Use this only to interpret unclear references (place names, "
            "historical events alluded to). Do NOT use it to infer facts "
            "not stated or clearly implied in the entry text itself."
        ),
    },
    # "example_book_2": { ... },
}



## 6. State summary

Built entirely from code (the registries + last known narrator state), never
generated by the model. This is what gets passed into the prompt each turn
so the model doesn't need to re-derive continuity itself.


In [ ]:

@dataclass
class NarratorState:
    last_known_location: str = None
    last_location_stated_entry: str = None
    travel_status: str = "not_traveling"   # in_transit | departed | arrived | not_traveling

def build_state_summary(narrator_state: NarratorState, char_registry: CharacterRegistry) -> dict:
    return {
        "narrator_last_known_location": narrator_state.last_known_location,
        "narrator_last_stated_in_entry": narrator_state.last_location_stated_entry,
        "narrator_travel_status": narrator_state.travel_status,
        "known_characters": [
            {
                "character_id": c.character_id,
                "label": c.canonical_label,
                "relation_to_narrator": c.relation_to_narrator,
                "last_known_location": c.last_known_location,
            }
            for c in char_registry._by_id.values()
        ],
    }



## 7. LLM extraction

The schema is enforced via a forced tool call (Anthropic tool use), which is
more reliable than asking for JSON in prose. `evidence` fields are required
for every claim so the grounding check in step 8 has something to verify.


In [ ]:

EXTRACTION_TOOL = {
    "name": "extract_entry_data",
    "description": "Extract structured location/travel/character data from one diary entry.",
    "input_schema": {
        "type": "object",
        "properties": {
            "narrator_location_w_writing": {
                "type": ["object", "null"],
                "properties": {
                    "label": {"type": "string", "description": "Take from the provided location list where possible; otherwise a short label."},
                    "evidence": {"type": "string", "description": "Verbatim quote from THIS entry supporting the location."},
                },
                "required": ["label", "evidence"],
                "description": "Null if this entry does not state or clearly imply where the narrator is while writing.",
            },
            "travel_status": {
                "type": "string",
                "enum": ["in_transit", "departed", "arrived", "not_traveling"],
            },
            "travel_status_evidence": {
                "type": ["string", "null"],
                "description": "Verbatim quote supporting travel_status; null if travel_status is 'not_traveling' with no textual basis needed.",
            },
            "character_mentions": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "label": {"type": "string", "description": "Take from the provided name list where possible."},
                        "character_location": {
                            "type": ["object", "null"],
                            "properties": {
                                "label": {"type": "string"},
                                "evidence": {"type": "string"},
                            },
                            "required": ["label", "evidence"],
                        },
                    },
                    "required": ["label"],
                },
            },
            "all_place_mentions": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "label": {"type": "string"},
                        "evidence": {"type": "string"},
                    },
                    "required": ["label", "evidence"],
                },
            },
        },
        "required": ["narrator_location_w_writing", "travel_status", "character_mentions", "all_place_mentions"],
    },
}

EXTRACTION_SYSTEM_PROMPT = """You are annotating one diary entry from a work \
of children's historical fiction, as part of a research project on how \
Canadian history is represented in these books.

Rules:
- Only report what THIS ENTRY states or clearly, unambiguously implies.
  If the entry does not mention the narrator's location, set
  narrator_location_w_writing to null rather than guessing from prior context.
- Every evidence field must be a VERBATIM quote copied exactly from the
  entry text provided below (not paraphrased, not from other entries).
- Prefer labels from the provided candidate name/location lists when the
  entity matches one; only introduce a new label if it's clearly a new
  entity not on those lists.
- Do not use outside knowledge of the book's plot to fill in gaps -- if it
  isn't in the text, it isn't reported.
- The "basic_context" and "state_summary" you're given are for
  disambiguating references (pronouns, nicknames, places already
  established), not sources of new facts.
"""

def build_prompt(entry_row, prior_entries_text, basic_context, state_summary,
                  candidate_names, candidate_places):
    return json.dumps({
        "basic_context": basic_context,
        "state_summary": state_summary,
        "candidate_character_names": candidate_names,
        "candidate_location_names": candidate_places,
        "prior_entries_for_reference_only": prior_entries_text,
        "current_entry_id": entry_row["entry_id"],
        "current_entry_text": entry_row["entry_text"],
    }, indent=2)

def call_llm_extract(prompt_payload: str, max_retries=3) -> dict:
    for attempt in range(max_retries):
        try:
            resp = client.messages.create(
                model=MODEL_NAME,
                max_tokens=1500,
                system=EXTRACTION_SYSTEM_PROMPT,
                tools=[EXTRACTION_TOOL],
                tool_choice={"type": "tool", "name": "extract_entry_data"},
                messages=[{"role": "user", "content": prompt_payload}],
            )
            for block in resp.content:
                if block.type == "tool_use":
                    return block.input
            raise ValueError("No tool_use block in response")
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)



## 8. Grounding validation

Checks every `evidence` string against the actual entry text. `exact` match
requires the quote to be a verbatim substring; `fuzzy` uses a similarity
ratio to catch near-verbatim paraphrase (still flagged, since your rule is
verbatim quoting, but useful to distinguish "slightly reworded" from
"invented entirely" when triaging QA failures).


In [ ]:

FUZZY_MATCH_THRESHOLD = 0.85

def check_grounding(evidence: str, entry_text: str) -> dict:
    if not evidence:
        return {"status": "missing", "similarity": 0.0}
    if evidence in entry_text:
        return {"status": "exact", "similarity": 1.0}
    # sliding fuzzy match against windows of the entry text roughly the
    # length of the evidence string
    best = 0.0
    window = max(len(evidence), 20)
    for i in range(0, max(len(entry_text) - window, 1), 10):
        chunk = entry_text[i:i + window + 20]
        ratio = difflib.SequenceMatcher(None, evidence, chunk).ratio()
        best = max(best, ratio)
    status = "fuzzy" if best >= FUZZY_MATCH_THRESHOLD else "ungrounded"
    return {"status": status, "similarity": round(best, 3)}

def validate_entry_grounding(extraction: dict, entry_text: str) -> dict:
    """Walks the extraction dict, checking every evidence field, and returns
    a QA report alongside a pass/fail flag for the whole entry."""
    report = {"fields": [], "all_grounded": True}

    def _check(field_name, evidence):
        result = check_grounding(evidence, entry_text)
        report["fields"].append({"field": field_name, "evidence": evidence, **result})
        if result["status"] == "ungrounded":
            report["all_grounded"] = False

    nl = extraction.get("narrator_location_w_writing")
    if nl:
        _check("narrator_location_w_writing", nl.get("evidence", ""))

    if extraction.get("travel_status_evidence"):
        _check("travel_status_evidence", extraction["travel_status_evidence"])

    for cm in extraction.get("character_mentions", []):
        cl = cm.get("character_location")
        if cl:
            _check(f"character_mentions[{cm.get('label')}]", cl.get("evidence", ""))

    for pm in extraction.get("all_place_mentions", []):
        _check(f"all_place_mentions[{pm.get('label')}]", pm.get("evidence", ""))

    return report



## 9. Main pipeline loop

Processes one book at a time, entry by entry, in narrative order —
maintaining the registries and narrator state as it goes, calling the LLM,
validating grounding, and updating state for the next iteration.


In [ ]:

def process_book(book_code: str, df: pd.DataFrame,
                  reviewed_entities: dict) -> tuple:
    char_registry = CharacterRegistry()
    char_registry.seed_from_reviewed(reviewed_entities.get("characters", []))
    loc_registry = LocationRegistry()
    loc_registry.seed_from_reviewed(reviewed_entities.get("locations", []))

    narrator_state = NarratorState()
    basic_context = BASIC_CONTEXT.get(book_code, {})

    results = []
    qa_reports = []
    entry_texts_in_order = []  # for building the rolling context window

    for idx, row in df.iterrows():
        entry_texts_in_order.append({"entry_id": row["entry_id"], "text": row["entry_text"]})
        prior_context = entry_texts_in_order[-(CONTEXT_WINDOW_ENTRIES + 1):-1]

        candidate_names = list(char_registry._alias_to_id.keys())
        candidate_places = list(loc_registry._alias_to_id.keys())

        state_summary = build_state_summary(narrator_state, char_registry)
        prompt = build_prompt(row, prior_context, basic_context, state_summary,
                               candidate_names, candidate_places)

        extraction = call_llm_extract(prompt)
        qa = validate_entry_grounding(extraction, row["entry_text"])
        qa["entry_id"] = row["entry_id"]
        qa_reports.append(qa)

        # --- update registries & narrator state from this entry's output ---
        nl = extraction.get("narrator_location_w_writing")
        if nl and nl.get("label"):
            resolved = loc_registry.register_or_flag(nl["label"], row["entry_id"])
            narrator_state.last_known_location = resolved.canonical_label
            narrator_state.last_location_stated_entry = row["entry_id"]
        narrator_state.travel_status = extraction.get("travel_status", narrator_state.travel_status)

        resolved_mentions = []
        for cm in extraction.get("character_mentions", []):
            char_entry = char_registry.register_or_flag(cm["label"], row["entry_id"])
            loc_label = None
            if cm.get("character_location") and cm["character_location"].get("label"):
                resolved_loc = loc_registry.register_or_flag(
                    cm["character_location"]["label"], row["entry_id"])
                loc_label = resolved_loc.canonical_label
                char_registry.update_location(char_entry.character_id, loc_label, row["entry_id"])
            resolved_mentions.append({
                "character_id": char_entry.character_id,
                "label": char_entry.canonical_label,
                "character_location": loc_label,       # None if not stated THIS entry
                "_evidence": (cm.get("character_location") or {}).get("evidence"),
            })

        resolved_places = []
        for pm in extraction.get("all_place_mentions", []):
            resolved_loc = loc_registry.register_or_flag(pm["label"], row["entry_id"])
            resolved_places.append({"label": resolved_loc.canonical_label, "_evidence": pm.get("evidence")})

        results.append({
            "entry_id": row["entry_id"],
            "narrator_location_w_writing": narrator_state.last_known_location if nl else None,
            "narrator_location_stated_this_entry": bool(nl),
            "_narrator_location_evidence": nl.get("evidence") if nl else None,
            "travel_status": extraction.get("travel_status"),
            "_travel_status_evidence": extraction.get("travel_status_evidence"),
            "character_mentions": resolved_mentions,
            "all_place_mentions": resolved_places,
        })

    return results, qa_reports, char_registry, loc_registry



## 10. Carry-forward post-processing & export

Fills `narrator_location_w_writing` forward for entries where it wasn't
stated (deterministic, auditable — you can see exactly which entries were
carried forward via `narrator_location_stated_this_entry`). Then writes
two versions of the output: a full **debug** file (with evidence + grounding
flags, for comparing against your manual gold labels) and a lean **public**
file (evidence stripped, matching the schema intended for the site).


In [ ]:

def carry_forward_locations(results: list) -> list:
    last_known = None
    for r in results:
        if r["narrator_location_w_writing"]:
            last_known = r["narrator_location_w_writing"]
        else:
            r["narrator_location_w_writing"] = last_known
    return results

def export_book(book_code, results, qa_reports, char_registry, loc_registry):
    results = carry_forward_locations(results)

    # --- debug export (keep everything, for eval against gold labels) ---
    with open(DEBUG_DIR / f"{book_code}_debug.json", "w") as f:
        json.dump(results, f, indent=2)
    with open(DEBUG_DIR / f"{book_code}_qa_report.json", "w") as f:
        json.dump(qa_reports, f, indent=2)

    # --- public export (strip underscore-prefixed evidence/debug fields) ---
    def strip_debug(obj):
        if isinstance(obj, dict):
            return {k: strip_debug(v) for k, v in obj.items() if not k.startswith("_")}
        if isinstance(obj, list):
            return [strip_debug(x) for x in obj]
        return obj

    public_results = strip_debug(results)
    with open(OUT_DIR / f"{book_code}_public.json", "w") as f:
        json.dump(public_results, f, indent=2)

    # --- registries ---
    with open(REGISTRY_DIR / f"{book_code}_characters.json", "w") as f:
        json.dump(char_registry.export(), f, indent=2)
    with open(REGISTRY_DIR / f"{book_code}_locations.json", "w") as f:
        json.dump(loc_registry.export(), f, indent=2)

    n_flagged = sum(1 for q in qa_reports if not q["all_grounded"])
    print(f"[{book_code}] {len(results)} entries processed. "
          f"{n_flagged} entries have at least one ungrounded evidence field. "
          f"{len(char_registry.pending_review)} characters and "
          f"{len(loc_registry.pending_review)} locations were auto-added "
          f"mid-book and should be reviewed.")

    return public_results



## 11. Run the pilot

Run this after: (a) CSVs are in `./data/`, (b) `BASIC_CONTEXT` is filled in
for both pilot books, (c) `*_reviewed.json` candidate lists exist for both
pilot books in `output/candidate_lists/`.


In [ ]:

# books = load_pilot_books()
#
# for book_code, df in books.items():
#     candidates = ner_pass_book(df)
#     save_candidate_lists(book_code, candidates)
#
# --- pause here: review/create the *_reviewed.json files for each book ---
#
# for book_code, df in books.items():
#     reviewed = load_reviewed_entities(book_code)
#     results, qa_reports, char_reg, loc_reg = process_book(book_code, df, reviewed)
#     export_book(book_code, results, qa_reports, char_reg, loc_reg)



## 12. Spot-check against manual gold labels

A minimal side-by-side printer to speed up comparing pipeline output against
your manual annotations while you're evaluating pilot results. Extend this
into proper precision/recall scoring (per field, as discussed in planning)
once you have a gold label file to compare against systematically.


In [ ]:

def spot_check(book_code: str, entry_id: str, gold: dict = None):
    with open(DEBUG_DIR / f"{book_code}_debug.json") as f:
        debug_data = {r["entry_id"]: r for r in json.load(f)}
    with open(DEBUG_DIR / f"{book_code}_qa_report.json") as f:
        qa_data = {q["entry_id"]: q for q in json.load(f)}

    pred = debug_data.get(entry_id)
    qa = qa_data.get(entry_id)
    print(f"--- {entry_id} ---")
    print("PREDICTED:", json.dumps(pred, indent=2))
    if qa:
        print("GROUNDING:", json.dumps(qa, indent=2))
    if gold:
        print("GOLD:", json.dumps(gold, indent=2))

# Example:
# spot_check("SofS", "SofS_ch1_e1")



## Notes for the pilot review

Things worth specifically checking once the pilot books have run, tying back
to the open questions from planning:

- **Retrospective narrative gap**: for any book flagged `"retrospective"` in
  `BASIC_CONTEXT`, check whether `narrator_location_w_writing` correctly
  stays fixed (e.g. in Canada) while the entry *narrates* events elsewhere.
  If the model conflates the two, that's the point at which a
  `narrated_event_location` field becomes worth adding — but only add it if
  the pilot shows the problem is real and frequent, not preemptively.
- **Carry-forward accuracy**: spot-check entries where
  `narrator_location_stated_this_entry` is `False` — is the carried-forward
  location actually still correct, or did an off-page move happen that the
  text never states outright?
- **Grounding pass rate**: the fraction of `ungrounded` evidence fields in
  `qa_report.json` is your headline hallucination metric. Investigate any
  systematic pattern (e.g. always failing on `character_location`, never on
  `narrator_location`) before scaling up.
- **`pending_review` lists**: entities the registries auto-added mid-book
  weren't in your reviewed candidate lists — check whether they're real
  entities the NER pass missed, or a sign the candidate-list review step
  needs another pass.
- **Cost/rate**: at pilot scale this is a handful of dollars; before running
  the full ~35-book corpus, multiply the pilot's per-entry token usage by
  your total entry count to get a realistic estimate, and consider whether
  entries that fail grounding checks warrant a second, more careful pass
  with a larger model rather than raising the model tier for every entry.
